In [1]:
from datetime import datetime, timedelta
from io import StringIO

import lxml
import pandas as pd
import requests
from lxml import html

In [2]:
def format_date(date_datetime):
    return str(int(date_datetime.timestamp()))


def subdomain_div_history(symbol, start, end):
    format_url = "{0}/history?period1={1}&period2={2}"
    tail_url = "&interval=div%7Csplit&filter=div&frequency=1d"
    subdomain = format_url.format(symbol, start, end) + tail_url
    return subdomain


def subdomain_statistics(symbol):
    format_url = "{0}/key-statistics"
    return format_url.format(symbol)


def header(subdomain):
    hdrs = {
        "authority": "finance.yahoo.com",
        "method": "GET",
        "path": subdomain,
        "scheme": "https",
        "accept": "text/html,application/xhtml+xml",
        "accept-encoding": "gzip, deflate, br",
        "accept-language": "en-US,en;q=0.9",
        "cache-control": "no-cache",
        "dnt": "1",
        "pragma": "no-cache",
        "sec-fetch-mode": "navigate",
        "sec-fetch-site": "same-origin",
        "sec-fetch-user": "?1",
        "upgrade-insecure-requests": "1",
        "user-agent": "Mozilla/5.0",
    }
    return hdrs


def scrape_page(url, header):
    page = requests.get(url, headers=header)
    element_html = html.fromstring(page.content)
    table = element_html.xpath("//table")
    table_tree = lxml.etree.tostring(table[0], method="xml", encoding="unicode")
    panda = pd.read_html(StringIO(table_tree))
    return panda


def clean_dividends(symbol, dividends):
    dividends = pd.DataFrame(dividends)
    dividends = dividends.rename(columns={dividends.columns[1]: "Dividends"})
    dividends = dividends.drop(columns=dividends.columns[[2, 3, 4, 5, 6]])
    dividends["Dividends"] = dividends["Dividends"].str.replace("Dividend", "")
    dividends["Date"] = pd.to_datetime(dividends["Date"]).dt.strftime("%m-%d-%Y")
    dividends["Dividends"] = dividends["Dividends"].astype(float)
    dividends["Dividends"] = dividends["Dividends"].apply(lambda x: f"${x:,.2f}")
    return dividends


def clean_statistics(symbol, statistics):
    statistics = statistics.rename(columns={statistics.columns[0]: ""})
    statistics = statistics.reset_index(drop=True)
    return statistics

In [3]:
symbol = "VZ"
# create datetime objects
start = datetime.today() - timedelta(days=9125)  # noqa: DTZ002
end = datetime.today()  # noqa: DTZ002
# properly format the date to epoch time
start = format_date(start)
end = format_date(end)
# format the subdomain
sub_div = subdomain_div_history(symbol, start, end)
# customize the request header
hdrs_div = header(sub_div)

# concatenate the subdomain with the base URL
base_url = "https://finance.yahoo.com/quote/"
url = base_url + sub_div
# scrape the dividend history table from Yahoo Finance
dividends = scrape_page(url, hdrs_div)

# clean the dividend history table
clean_div = clean_dividends(symbol, dividends[0])

# format key statistics subdomain
sub_stats = subdomain_statistics(symbol)
hdrs_stats = header(sub_stats)
url_stats = base_url + sub_stats
key_statistics = scrape_page(url_stats, hdrs_stats)

stats = clean_statistics(symbol, key_statistics[0])

In [4]:
# if files already exist, overwrite them
clean_div.to_csv("Dividend_Data.csv", index=False, mode="w")
stats.to_csv("Key_Statistics_Data.csv", index=False, mode="w")